In [2]:
sealed trait Expr
case class Times(op1: Expr, op2: Expr) extends Expr
case class Plus(op1: Expr, op2: Expr) extends Expr
case class Div(op1: Expr, op2: Expr) extends Expr
case class Minus(op1: Expr, op2: Expr) extends Expr
case class Sqrt(op1: Expr) extends Expr
case class Minus1(op1: Expr) extends Expr
case class Num(i: Int) extends Expr


defined trait Expr
defined class Times
defined class Plus
defined class Div
defined class Minus
defined class Sqrt
defined class Minus1
defined class Num

## Parser (TBD)

In [11]:
import $ivy.`org.scala-lang.modules::scala-parser-combinators:2.1.0`

Downloaded https://repo1.maven.org/maven2/org/scala-lang/modules/scala-parser-combinators_2.13/2.1.0/scala-parser-combinators_2.13-2.1.0.pom
Downloaded https://repo1.maven.org/maven2/org/scala-lang/modules/scala-parser-combinators_2.13/2.1.0/scala-parser-combinators_2.13-2.1.0-sources.jar
Downloaded https://repo1.maven.org/maven2/org/scala-lang/modules/scala-parser-combinators_2.13/2.1.0/scala-parser-combinators_2.13-2.1.0.jar


import $ivy.$                                                       

In [12]:
import scala.util.parsing.combinator._

import scala.util.parsing.combinator._

## Serializer

In [3]:
def unparse(e: Expr): String = 
    e match {
        case Times(e1, e2) => s"(${unparse(e1)}*${unparse(e2)})"
        case Plus(e1, e2) => s"(${unparse(e1)}+${unparse(e2)})"
        case Div(e1, e2) => s"(${unparse(e1)}:${unparse(e2)})"
        case Minus(e1, e2) => s"(${unparse(e1)}-${unparse(e2)})"
        case Sqrt(e1) => s"sqrt(${unparse(e1)})"
        case Minus1(e1) => s"-(${unparse(e1)})"
        case Num(n) => n.toString
    }

defined function unparse

## step-by-step evaluation

In [4]:
import scala.math.sqrt

import scala.math.sqrt

In [8]:
def eval1(e: Expr): (Expr => Expr) => List[Expr] => (List[Expr], Int) = ctx => out => 
    e match {
        case Times(e1, e2) => 
            val (out1, n1) = eval1(e1)(e => ctx(Times(e, e2)))(out)
            val (out2, n2) = eval1(e2)(e => ctx(Times(Num(n1), e)))(out1)
            (out2 :+ ctx(Num(n1*n2)), n1*n2) 
        case Plus(e1, e2) => 
            val (out1, n1) = eval1(e1)(e => ctx(Plus(e, e2)))(out)
            val (out2, n2) = eval1(e2)(e => ctx(Plus(Num(n1), e)))(out1)
            (out2 :+ ctx(Num(n1+n2)), n1+n2) 
        case Div(e1, e2) => 
            val (out1, n1) = eval1(e1)(e => ctx(Div(e, e2)))(out)
            val (out2, n2) = eval1(e2)(e => ctx(Div(Num(n1), e)))(out1)
            (out2 :+ ctx(Num(n1/n2)), n1/n2) 
        case Minus(e1, e2) => 
            val (out1, n1) = eval1(e1)(e => ctx(Minus(e, e2)))(out)
            val (out2, n2) = eval1(e2)(e => ctx(Minus(Num(n1), e)))(out1)
            (out2 :+ ctx(Num(n1-n2)), n1-n2)
        case Sqrt(e) => 
            val (out1, n1) = eval1(e)(e => ctx(Sqrt(e)))(out)
            (out1 :+ ctx(Num(sqrt(n1).toInt)), sqrt(n1).toInt)
        case Minus1(e) =>
            val (out1, n1) = eval1(e)(e => ctx(Minus1(e)))(out)
            (out1 :+ ctx(Num(-n1)), -n1)
        case Num(n) => 
            (out, n)
    }

defined function eval1

In [9]:
def eval(e: Expr): List[Expr] = 
    eval1(e)(identity)(List(e))._1

defined function eval

### use cases 

In [10]:
val e1: Expr = Div(Sqrt(Times(Plus(Num(1),Num(2)), Num(3))), Num(4))

e1: Expr = Div(Sqrt(Times(Plus(Num(1), Num(2)), Num(3))), Num(4))

In [13]:
eval(e1) foreach { (unparse _) andThen println }

(sqrt(((1+2)*3)):4)
(sqrt((3*3)):4)
(sqrt(9):4)
(3:4)
0
